# Run Large Models in Notebook with Public Proxy URL

This notebook builds and launches the **MoE Stream Engine** (or an optional **Ollama / llama.cpp** backend) to run large GGUF / MoE models directly inside a notebook environment (Google Colab, Kaggle, Jupyter, or Codespaces) and exposes the server via a **Public HTTPS Proxy URL**.

### What You Get
- **Public Web UI URL**: Full interactive workspace (`/`) accessible from any browser.
- **OpenAI-Compatible API URL**: `https://<public-proxy-url>/v1/chat/completions` (supports SSE token streaming & dual-channel reasoning).
- **Telemetry & Diagnostics**: `/health`, `/profile`, `/experts`, `/v1/systemone`, and `/v1/images/generations`.
- **Multiple Public Tunnel Providers**:
  1. **Cloudflare Quick Tunnel (`cloudflared`)** — Free, fast HTTPS public URL (`https://*.trycloudflare.com`), no account or token required.
  2. **Pinggy HTTPS Tunnel** — Automatic zero-config fallback (`https://*.a.free.pinggy.link`).
  3. **Google Colab Proxy** — Native Colab proxy URL if running on Google Colab.
  4. **ngrok** — Optional if you provide an `NGROK_AUTH_TOKEN`.

In [ ]:
# @title 1. Setup Environment, Install Tunnel Binary & Compile MoE Engine
import os
import platform
import shutil
import subprocess
import urllib.request
from pathlib import Path

WORKSPACE_DIR = Path(os.getcwd()).resolve()
print(f"[INFO] Workspace directory: {WORKSPACE_DIR}")

# 1. Ensure Python requests & huggingface_hub are available
subprocess.run(["pip", "install", "-q", "requests", "huggingface_hub", "openai"], check=False)

# 2. Download cloudflared binary for zero-config public HTTPS proxy URLs
CLOUDFLARED_BIN = WORKSPACE_DIR / "cloudflared"
if shutil.which("cloudflared"):
    CLOUDFLARED_BIN = Path(shutil.which("cloudflared"))
elif not CLOUDFLARED_BIN.exists():
    arch = platform.machine().lower()
    cf_arch = "arm64" if arch in ("aarch64", "arm64") else "amd64"
    cf_url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}"
    print(f"[SETUP] Downloading cloudflared from {cf_url}...")
    urllib.request.urlretrieve(cf_url, str(CLOUDFLARED_BIN))
    os.chmod(CLOUDFLARED_BIN, 0o755)
print(f"[OK] cloudflared binary ready at: {CLOUDFLARED_BIN}")

# 3. Check for repository files or auto-clone in Colab / Kaggle environments
if not (WORKSPACE_DIR / "CMakeLists.txt").exists():
    repo_url = "https://github.com/jacxishipping/jxai.git"
    repo_target = WORKSPACE_DIR / "jxai"
    if not repo_target.exists():
        print(f"[SETUP] Cloning jxai repository into {repo_target}...")
        subprocess.run(["git", "clone", repo_url, str(repo_target)], check=True)
    os.chdir(repo_target)
    WORKSPACE_DIR = repo_target
    print(f"[INFO] Switched active workspace to: {WORKSPACE_DIR}")

# 4. Compile native C++20 MoE Stream Engine
ENGINE_BIN = WORKSPACE_DIR / "build" / "moe_engine"
build_dir = WORKSPACE_DIR / "build"
build_dir.mkdir(exist_ok=True)
if not ENGINE_BIN.exists():
    print("[BUILD] Compiling MoE Stream Engine (C++20 AVX2/FMA + OpenMP)...")
    subprocess.run(["cmake", "-S", str(WORKSPACE_DIR), "-B", str(build_dir), "-DCMAKE_BUILD_TYPE=Release"], check=True)
    subprocess.run(["cmake", "--build", str(build_dir), "-j"], check=True)
print(f"[OK] MoE Stream Engine binary ready: {ENGINE_BIN}")
subprocess.run([str(ENGINE_BIN), "doctor"], check=False)


In [ ]:
# @title 2. Configure & Download Large Model (GGUF / Hugging Face)
import os
import subprocess
from pathlib import Path

# Choose Backend:
# - "moe_engine": Uses this repository's C++20 MoE Stream Engine (PILOT NVMe streaming + LFRU caching + Web UI)
# - "ollama":     Uses Ollama to pull and serve large models (e.g. qwen2.5:14b, deepseek-r1:14b, mixtral:8x7b)
BACKEND = os.environ.get("MODEL_BACKEND", "moe_engine")  # "moe_engine" or "ollama"

# Model configuration for MoE Stream Engine:
# Leave HF_MODEL_REPO_OR_URL empty to use the local/forged GGUF model, or specify any Hugging Face GGUF repo/URL:
# Examples:
#   "Qwen/Qwen2.5-0.5B-Instruct-GGUF:qwen2.5-0.5b-instruct-q4_0.gguf"
#   "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_0.gguf"
HF_MODEL_REPO_OR_URL = ""  # Set to a Hugging Face GGUF URL or repo:filename to download a large model
LOCAL_MODEL_PATH = "models/qwen_run.gguf"

# Model name if using BACKEND == "ollama":
OLLAMA_MODEL = "qwen2.5:7b"

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

if BACKEND == "moe_engine":
    engine_bin = Path("build/moe_engine")
    if HF_MODEL_REPO_OR_URL.strip():
        print(f"[PULL] Downloading model from Hugging Face: {HF_MODEL_REPO_OR_URL}")
        subprocess.run([str(engine_bin), "pull", HF_MODEL_REPO_OR_URL, "--out", str(models_dir)], check=True)
        # Pick the newest .gguf file in models/
        gguf_files = sorted(models_dir.glob("*.gguf"), key=lambda p: p.stat().st_mtime, reverse=True)
        if gguf_files:
            LOCAL_MODEL_PATH = str(gguf_files[0])
    elif not Path(LOCAL_MODEL_PATH).exists():
        print(f"[FORGE] {LOCAL_MODEL_PATH} not found; forging operational MoE GGUF model...")
        subprocess.run([str(engine_bin), "forge", "--out", LOCAL_MODEL_PATH], check=True)

    model_size_mb = Path(LOCAL_MODEL_PATH).stat().st_size / (1024 * 1024)
    print(f"[READY] Selected Model: {LOCAL_MODEL_PATH} ({model_size_mb:.2f} MB)")
elif BACKEND == "ollama":
    if not shutil.which("ollama"):
        print("[SETUP] Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print(f"[READY] Ollama configured for model: {OLLAMA_MODEL}")

In [ ]:
# @title 3. Start Inference Server & Create Public Proxy URL
import os
import re
import time
import socket
import subprocess
import threading
import requests
from pathlib import Path

PORT = int(os.environ.get("PORT", 8000))
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")  # Optional: set if you prefer ngrok

def is_port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

# 1. Start the inference server on localhost:PORT
if not is_port_in_use(PORT):
    if BACKEND == "moe_engine":
        cmd = [
            str(Path("build/moe_engine").resolve()),
            "serve",
            "--model", str(LOCAL_MODEL_PATH),
            "--port", str(PORT),
            "--web-dir", "web/dist"
        ]
        print(f"[SERVER] Starting MoE Stream Engine on port {PORT}: {' '.join(cmd)}")
        server_log = open("moe_server.log", "w")
        server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
    else:
        env = os.environ.copy()
        env["OLLAMA_HOST"] = f"0.0.0.0:{PORT}"
        print(f"[SERVER] Starting Ollama server on port {PORT}...")
        server_log = open("ollama_server.log", "w")
        server_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=server_log, stderr=subprocess.STDOUT)
        time.sleep(3)
        subprocess.run(["ollama", "pull", OLLAMA_MODEL], env=env, check=True)

    # Wait for server readiness
    for _ in range(30):
        if is_port_in_use(PORT):
            break
        time.sleep(0.5)
    print(f"[OK] Local server is listening on http://127.0.0.1:{PORT}")
else:
    print(f"[OK] Server already running on http://127.0.0.1:{PORT}")

# 2. Start Public HTTPS Tunnel & Extract Public Proxy URL
PUBLIC_URL = None
COLAB_PROXY_URL = None

# Check if running inside Google Colab for native Colab proxy URL
try:
    from google.colab import output as colab_output
    COLAB_PROXY_URL = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception:
    pass

def start_cloudflared_tunnel(port: int, timeout: int = 25):
    cf_bin = str(CLOUDFLARED_BIN) if "CLOUDFLARED_BIN" in globals() and Path(CLOUDFLARED_BIN).exists() else "cloudflared"
    proc = subprocess.Popen(
        [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    found_url = None
    start_time = time.time()

    def reader():
        nonlocal found_url
        for line in proc.stdout:
            match = url_pattern.search(line)
            if match and not found_url:
                found_url = match.group(0)

    t = threading.Thread(target=reader, daemon=True)
    t.start()
    while time.time() - start_time < timeout and not found_url:
        time.sleep(0.3)
    return found_url, proc

def start_pinggy_tunnel(port: int, timeout: int = 20):
    proc = subprocess.Popen(
        [
            "ssh", "-p", "443",
            "-R0:127.0.0.1:" + str(port),
            "-o", "StrictHostKeyChecking=no",
            "-o", "ServerAliveInterval=30",
            "a.pinggy.io"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    url_pattern = re.compile(r"https://[a-zA-Z0-9.-]+\.pinggy\.link")
    found_url = None
    start_time = time.time()

    def reader():
        nonlocal found_url
        for line in proc.stdout:
            match = url_pattern.search(line)
            if match and not found_url:
                found_url = match.group(0)

    t = threading.Thread(target=reader, daemon=True)
    t.start()
    while time.time() - start_time < timeout and not found_url:
        time.sleep(0.3)
    return found_url, proc

print("[TUNNEL] Creating public HTTPS proxy tunnel...")
PUBLIC_URL, tunnel_proc = start_cloudflared_tunnel(PORT)

if not PUBLIC_URL:
    print("[TUNNEL] cloudflared URL not acquired, trying Pinggy HTTPS tunnel fallback...")
    PUBLIC_URL, tunnel_proc = start_pinggy_tunnel(PORT)

if not PUBLIC_URL and COLAB_PROXY_URL:
    PUBLIC_URL = COLAB_PROXY_URL.rstrip("/")

print("\n" + "=" * 72)
if PUBLIC_URL:
    print(f"  PUBLIC PROXY URL READY:       {PUBLIC_URL}")
    print(f"  Web UI Workspace:             {PUBLIC_URL}/")
    print(f"  OpenAI API Base URL:          {PUBLIC_URL}/v1")
    print(f"  Chat Completions Endpoint:    {PUBLIC_URL}/v1/chat/completions")
    print(f"  Health & Hardware Endpoint:   {PUBLIC_URL}/health")
    print(f"  Profiler Telemetry Endpoint:  {PUBLIC_URL}/profile")
    print(f"  MoE Expert Routing Hitmap:    {PUBLIC_URL}/experts")
if COLAB_PROXY_URL:
    print(f"  Google Colab Native Proxy:    {COLAB_PROXY_URL}")
print(f"  Local Direct URL:             http://127.0.0.1:{PORT}")
print("=" * 72)

In [ ]:
# @title 4. Test Public Proxy URL (Health Check & OpenAI Chat Completion)
import json
import requests

target_base = (PUBLIC_URL or f"http://127.0.0.1:{PORT}").rstrip("/")
print(f"[TEST] Verifying server through: {target_base}\n")

# 1. Check /health
try:
    health_resp = requests.get(f"{target_base}/health", timeout=15)
    print("[1] GET /health ->", health_resp.status_code)
    print(json.dumps(health_resp.json(), indent=2))
except Exception as e:
    print(f"[WARN] /health check failed: {e}")

# 2. Test /v1/chat/completions
print("\n[2] POST /v1/chat/completions ->")
payload = {
    "model": "moe-stream" if BACKEND == "moe_engine" else OLLAMA_MODEL,
    "messages": [
        {"role": "user", "content": "Explain how asynchronous NVMe expert prefetching speeds up large MoE models."}
    ],
    "max_tokens": 64,
    "temperature": 0.7,
    "stream": False
}

try:
    chat_resp = requests.post(f"{target_base}/v1/chat/completions", json=payload, timeout=30)
    print("Status:", chat_resp.status_code)
    data = chat_resp.json()
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f"[WARN] Chat request failed: {e}")

# 3. Print ready-to-use client snippets
print("\n" + "-" * 72)
print("Use this Public Proxy URL from anywhere (Python OpenAI SDK or cURL):")
print("-" * 72)
print(f'''
# --- Python OpenAI SDK ---
from openai import OpenAI

client = OpenAI(
    base_url="{target_base}/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="moe-stream",
    messages=[{{"role": "user", "content": "Hello from remote client!"}}],
    max_tokens=128
)
print(response.choices[0].message.content)

# --- cURL ---
curl -X POST "{target_base}/v1/chat/completions" \\
  -H "Content-Type: application/json" \\
  -d '{{"model": "moe-stream", "messages": [{{"role": "user", "content": "Hello!"}}], "max_tokens": 64}}'
''')

In [ ]:
# @title 5. (Optional) Keep Notebook Alive & Tail Server Logs
import time
from pathlib import Path

log_file = Path("moe_server.log" if BACKEND == "moe_engine" else "ollama_server.log")
print(f"[KEEP-ALIVE] Public Proxy Active at: {PUBLIC_URL}")
print("Press Stop / Interrupt Kernel when you want to shut down the tunnel.\n")

try:
    last_pos = 0
    while True:
        if log_file.exists():
            with open(log_file, "r", errors="replace") as f:
                f.seek(last_pos)
                new_data = f.read()
                if new_data:
                    print(new_data, end="", flush=True)
                last_pos = f.tell()
        time.sleep(2)
except KeyboardInterrupt:
    print("\n[STOPPED] Keep-alive loop exited (server and tunnel are still running in background).")